<a href="https://colab.research.google.com/github/akshitasharmamca2025-jpg/Python_Assesment.py/blob/main/40_design_a_mini_student_management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Q40. Design a mini student management system using functions, dictionaries, file handling,
# exception handling, and Pandas for report generation.

import pandas as pd
import os

# --- Global Data Structure and File Configuration ---

# Dictionary to store student records. Key: Student ID (int), Value: Dictionary of student details.
# Example: {101: {'name': 'Alice', 'age': 20, 'grade': 'A', 'math': 85, 'science': 90}}
students_data = {}
DATA_FILE = 'students.csv'


# --- Helper Functions for File Handling ---

def load_data():
    """
    Loads student data from a CSV file into the global students_data dictionary.
    Handles FileNotFoundError if the file does not exist.
    """
    global students_data
    if not os.path.exists(DATA_FILE):
        print(f"Info: '{DATA_FILE}' not found. Starting with an empty student list.")
        students_data = {}
        return

    try:
        df = pd.read_csv(DATA_FILE, index_col='student_id')
        # Convert DataFrame back to dictionary format
        students_data = df.to_dict(orient='index')
        print(f"Success: Loaded {len(students_data)} student records from '{DATA_FILE}'.")
    except pd.errors.EmptyDataError:
        print(f"Info: '{DATA_FILE}' is empty. Starting with an empty student list.")
        students_data = {}
    except FileNotFoundError:
        # This should ideally be caught by os.path.exists but good for redundancy
        print(f"Error: '{DATA_FILE}' not found. Starting with an empty student list.")
        students_data = {}
    except Exception as e:
        print(f"Error loading data: {e}. Starting with an empty student list.")
        students_data = {}

def save_data():
    """
    Saves the global students_data dictionary to a CSV file.
    Converts the dictionary to a Pandas DataFrame first for easy CSV writing.
    """
    global students_data
    if not students_data:
        print("Info: No data to save. The student list is empty.")
        # Optionally, remove the file if it exists but there's no data
        if os.path.exists(DATA_FILE):
            os.remove(DATA_FILE)
            print(f"Removed empty data file '{DATA_FILE}'.")
        return

    try:
        df = pd.DataFrame.from_dict(students_data, orient='index')
        df.index.name = 'student_id'
        df.to_csv(DATA_FILE)
        print(f"Success: Saved {len(students_data)} student records to '{DATA_FILE}'.")
    except Exception as e:
        print(f"Error saving data: {e}")

# --- Core Student Management Functions ---

def get_valid_input(prompt, type_func, error_msg, validation_func=None):
    """
    Helper function to get validated user input.
    """
    while True:
        try:
            value = type_func(input(prompt))
            if validation_func and not validation_func(value):
                print(error_msg)
            else:
                return value
        except ValueError:
            print(error_msg)

def add_student():
    """
    Adds a new student record to the system.
    Prompts for ID, name, age, and marks in subjects.
    Includes validation for inputs.
    """
    print("\n--- Add New Student ---")
    student_id = get_valid_input(
        "Enter Student ID (e.g., 101): ", int,
        "Invalid ID. Please enter a unique positive integer.",
        lambda x: x > 0 and x not in students_data
    )

    name = input("Enter Student Name: ").strip().title()
    if not name:
        print("Student name cannot be empty.")
        return

    age = get_valid_input(
        "Enter Age (e.g., 18): ", int,
        "Invalid Age. Please enter a positive integer.",
        lambda x: 5 <= x <= 100 # Reasonable age range
    )

    # Example subjects for marks
    subjects = ['math', 'science', 'english']
    marks = {}
    for subject in subjects:
        mark = get_valid_input(
            f"Enter {subject.title()} Marks (0-100): ", int,
            "Invalid Mark. Please enter an integer between 0 and 100.",
            lambda x: 0 <= x <= 100
        )
        marks[subject] = mark

    students_data[student_id] = {'name': name, 'age': age, **marks}
    print(f"Student '{name}' (ID: {student_id}) added successfully.")
    save_data() # Save after each addition

def view_students():
    """
    Displays all student records in a formatted table.
    Uses Pandas DataFrame for better presentation.
    """
    print("\n--- View All Students ---")
    if not students_data:
        print("No student records available.")
        return

    df = pd.DataFrame.from_dict(students_data, orient='index')
    df.index.name = 'Student ID'
    print(df)

def update_student():
    """
    Updates an existing student's details (name, age, or marks).
    """
    print("\n--- Update Student Record ---")
    student_id = get_valid_input(
        "Enter Student ID to update: ", int,
        "Invalid ID. Please enter a positive integer.",
        lambda x: x > 0
    )

    if student_id not in students_data:
        print(f"Error: Student with ID {student_id} not found.")
        return

    student = students_data[student_id]
    print(f"Current details for Student ID {student_id}:")
    for key, value in student.items():
        print(f"  {key.replace('_', ' ').title()}: {value}")

    print("What would you like to update?")
    print("1. Name")
    print("2. Age")
    print("3. Marks")
    print("4. Cancel")

    choice = input("Enter your choice (1-4): ").strip()

    if choice == '1':
        new_name = input(f"Enter new name (current: {student['name']}): ").strip().title()
        if new_name:
            student['name'] = new_name
            print("Name updated.")
        else:
            print("Name not changed (empty input).")
    elif choice == '2':
        new_age = get_valid_input(
            f"Enter new age (current: {student['age']}): ", int,
            "Invalid Age. Please enter a positive integer.",
            lambda x: 5 <= x <= 100
        )
        if new_age is not None: # get_valid_input returns None if validation fails
            student['age'] = new_age
            print("Age updated.")
    elif choice == '3':
        subjects = ['math', 'science', 'english'] # Assume these are the subjects
        for subject in subjects:
            current_mark = student.get(subject, 'N/A')
            new_mark = get_valid_input(
                f"Enter new {subject.title()} Marks (current: {current_mark}, 0-100): ", int,
                "Invalid Mark. Please enter an integer between 0 and 100.",
                lambda x: 0 <= x <= 100
            )
            if new_mark is not None:
                student[subject] = new_mark
                print(f"{subject.title()} marks updated.")
    elif choice == '4':
        print("Update cancelled.")
    else:
        print("Invalid choice.")

    save_data() # Save after update

def delete_student():
    """
    Deletes a student record from the system.
    """
    print("\n--- Delete Student Record ---")
    student_id = get_valid_input(
        "Enter Student ID to delete: ", int,
        "Invalid ID. Please enter a positive integer.",
        lambda x: x > 0
    )

    if student_id in students_data:
        confirm = input(f"Are you sure you want to delete student ID {student_id} ({students_data[student_id]['name']})? (yes/no): ").lower()
        if confirm == 'yes':
            del students_data[student_id]
            print(f"Student with ID {student_id} deleted successfully.")
            save_data() # Save after deletion
        else:
            print("Deletion cancelled.")
    else:
        print(f"Error: Student with ID {student_id} not found.")

def generate_report():
    """
    Generates various reports using Pandas, such as average marks and top/bottom students.
    """
    print("\n--- Generate Student Report ---")
    if not students_data:
        print("No student records available to generate a report.")
        return

    df = pd.DataFrame.from_dict(students_data, orient='index')
    df.index.name = 'Student ID'

    # Ensure marks columns exist before calculation
    subjects_with_marks = [s for s in ['math', 'science', 'english'] if s in df.columns]

    if not subjects_with_marks:
        print("No subject marks found in the data to generate a report.")
        print("DataFrame structure:")
        print(df.head())
        return

    # Calculate Total Marks and Average Marks
    df['Total Marks'] = df[subjects_with_marks].sum(axis=1)
    df['Average Marks'] = df[subjects_with_marks].mean(axis=1).round(2)

    print("\n--- Overall Student Performance ---")
    print(df[['name', 'Total Marks', 'Average Marks']].sort_values(by='Average Marks', ascending=False))

    print("\n--- Subject-wise Average Marks ---")
    print(df[subjects_with_marks].mean().round(2))

    print("\n--- Top 3 Students by Average Marks ---")
    print(df.nlargest(3, 'Average Marks')[['name', 'Average Marks']])

    print("\n--- Bottom 3 Students by Average Marks ---")
    print(df.nsmallest(3, 'Average Marks')[['name', 'Average Marks']])

    # You can add more complex reports here, e.g., students below a certain grade, etc.

# --- Main Menu Function ---

def main_menu():
    """
    Displays the main menu and handles user choices for the student management system.
    """
    load_data() # Load data when the program starts

    while True:
        print("\n--- Student Management System ---")
        print("1. Add Student")
        print("2. View All Students")
        print("3. Update Student Record")
        print("4. Delete Student Record")
        print("5. Generate Report")
        print("6. Exit")
        print("---------------------------------")

        choice = input("Enter your choice (1-6): ").strip()

        if choice == '1':
            add_student()
        elif choice == '2':
            view_students()
        elif choice == '3':
            update_student()
        elif choice == '4':
            delete_student()
        elif choice == '5':
            generate_report()
        elif choice == '6':
            print("Exiting Student Management System. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 6.")

# --- Run the System ---
if __name__ == "__main__":
    main_menu()


Info: 'students.csv' not found. Starting with an empty student list.

--- Student Management System ---
1. Add Student
2. View All Students
3. Update Student Record
4. Delete Student Record
5. Generate Report
6. Exit
---------------------------------
Enter your choice (1-6): 1

--- Add New Student ---
Enter Student ID (e.g., 101): 105
Enter Student Name: Akshita Sharma
Enter Age (e.g., 18): 20
Enter Math Marks (0-100): 98
Enter Science Marks (0-100): 97
Enter English Marks (0-100): 76
Student 'Akshita Sharma' (ID: 105) added successfully.
Success: Saved 1 student records to 'students.csv'.

--- Student Management System ---
1. Add Student
2. View All Students
3. Update Student Record
4. Delete Student Record
5. Generate Report
6. Exit
---------------------------------
Enter your choice (1-6): 6
Exiting Student Management System. Goodbye!
